# Day1: Count Matrix QC

このノートブックは Google Colab でそのまま実行できる Day1 用教材です。

目的:
- count matrix を読む
- サンプルごとの総カウント数を確認する
- treated で増えていそうな遺伝子を見つける

## 最初に: RNA-seq は何をしているのか

細胞の中では、DNAに保存された遺伝子情報が、必要に応じてRNAとして読み出されます。

ざっくり言うと、ある遺伝子のRNAが多いほど、その遺伝子がその細胞でよく使われている可能性があります。

RNA-seqは、細胞中のRNAを読んで、どの遺伝子由来のRNAがどれくらいあったかを数える実験です。

例:

- `TP53` 由来のRNA断片が 120 個見つかった
- `IL6` 由来のRNA断片が 30 個見つかった

このような数を、遺伝子ごと・サンプルごとに並べた表が `count matrix` です。

## count matrix とは

`count matrix` は、行が遺伝子、列がサンプル、値がRNA断片の数になっている表です。

| gene | control_1 | control_2 | treated_1 | treated_2 |
|---|---:|---:|---:|---:|
| TP53 | 120 | 100 | 300 | 280 |
| MYC | 300 | 280 | 600 | 650 |
| IL6 | 30 | 25 | 200 | 220 |

読み方:

- `gene`: 遺伝子名です。
- `control_1`, `control_2`: 比較用サンプルです。
- `treated_1`, `treated_2`: 薬剤処理など、条件を変えたサンプルです。
- 数字: その遺伝子に対応するRNA断片が何個読まれたかです。

例えば `IL6` は control では少なく、treated では多いので、treated で増えていそうだと考えます。

ただし、この数字だけで最終結論にはしません。サンプルごとに読まれた総量が違うため、後で正規化と統計解析が必要です。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 1. count matrix を作る

今回は学習用の小さなデータを使います。

In [ ]:
counts = pd.DataFrame({
    "gene": ["TP53", "MYC", "GAPDH", "ACTB", "IL6"],
    "control_1": [120, 300, 5000, 4200, 30],
    "control_2": [100, 280, 5100, 4000, 25],
    "treated_1": [300, 600, 5300, 4100, 200],
    "treated_2": [280, 650, 5200, 4300, 220],
}).set_index("gene")

counts

## 2. CSV として保存する

Colab 上で、解析結果をファイルに残す練習をします。

In [ ]:
output_dir = Path("day1_outputs")
figure_dir = output_dir / "figures"
output_dir.mkdir(exist_ok=True)
figure_dir.mkdir(exist_ok=True)

counts_path = output_dir / "toy_counts.csv"
counts.to_csv(counts_path)
counts_path

## 3. CSV を読み込む

In [ ]:
loaded_counts = pd.read_csv(counts_path, index_col=0)
loaded_counts

## 4. サンプルごとの総カウント数を確認する

RNA-seq ではサンプルごとに読まれた量が違うので、最初にここを見ます。

In [ ]:
library_size = loaded_counts.sum(axis=0)
library_size

In [ ]:
plt.figure(figsize=(6, 4))
library_size.plot(kind="bar", color=["#4C78A8", "#4C78A8", "#F58518", "#F58518"])
plt.ylabel("Total counts")
plt.title("Library size per sample")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

figure_path = figure_dir / "day1_library_size.png"
plt.savefig(figure_path, dpi=150)
plt.show()

figure_path

## 5. treated で増えていそうな遺伝子を探す

ここでは平均値の比を使って、ざっくり候補を見ます。

In [ ]:
treated_mean = loaded_counts[["treated_1", "treated_2"]].mean(axis=1)
control_mean = loaded_counts[["control_1", "control_2"]].mean(axis=1)
fold_change = (treated_mean + 1) / (control_mean + 1)
top_gene = fold_change.sort_values(ascending=False).index[0]

print(f"Treatedで増えていそうな遺伝子: {top_gene}")
print(
    f"理由: control平均={control_mean[top_gene]:.1f}, "
    f"treated平均={treated_mean[top_gene]:.1f}, "
    f"fold change={fold_change[top_gene]:.2f}"
)

## 今日の課題

1. `MYC` の control 平均と treated 平均を自分で計算する
2. `IL6` が候補になる理由を 1 文で説明する
3. 図を見て、どのサンプルが最も総カウント数が多いか答える